Understand how LLMs process text as tokens, how context windows
limit the amount of information a model can process, and how
token usage affects API cost.

In [1]:
text = "FastAPI is a modern Python framework for building APIs."

Characters = len(text)
Words = len(text.split())

print(f"Characters: {Characters}")
print(f"Words: {Words}")

Characters: 55
Words: 9


Count actual tokens

In [2]:
!pip -q install tiktoken

In [3]:
import tiktoken

text = "FastAPI is a modern Python framework for building APIs."

encoding = tiktoken.get_encoding("cl100k_base")

tokens = encoding.encode(text)

print("Text:", text)
print("Characters:",len(text))
print("Words:",len(text.split()))
print("Tokens:",len(tokens))
print("Token IDs:",tokens)

Text: FastAPI is a modern Python framework for building APIs.
Characters: 55
Words: 9
Tokens: 11
Token IDs: [33274, 7227, 374, 264, 6617, 13325, 12914, 369, 4857, 34456, 13]


## Context Window

A context window is the amount of tokenized information a model can
process in a single request.

The context can include:
- System instructions
- User prompt
- Conversation history
- Retrieved RAG documents

In RAG systems, retrieving too many chunks can increase cost,
latency, and the risk of exceeding the model's context limit.

Therefore, retrieval quality and context size are important parts
of building efficient AI applications.


Calculate LLM Cost

In [5]:
input_tokens = 2000
output_tokens = 500

input_price_per_million = 0.50
output_price_per_million = 1.50

input_cost = (input_tokens / 1_000_000) * input_price_per_million
output_cost = (output_tokens / 1_000_000) * output_price_per_million

total_cost = input_cost + output_cost

print("Input cost: $", input_cost)
print("Output cost: $", output_cost)
print("Total cost: $", total_cost)

Input cost: $ 0.001
Output cost: $ 0.00075
Total cost: $ 0.00175


Inspect Gemini's actual token usage

In [6]:
!pip -q install langchain-google-genai

ERROR: Could not find a version that satisfies the requirement colab-userdata (from versions: none)
ERROR: No matching distribution found for colab-userdata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 14.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [8]:
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=GEMINI_API_KEY,
    temperature=0
)

print("LLM Sucessfully Initialized")

LLM Sucessfully Initialized


In [10]:
response = llm.invoke(
    "Explain what FastAPI is in 3 sentences."
)

print(response.content)

print("\nToken Usage:")
print(response.usage_metadata)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': '**FastAPI** is a modern, high-performance web framework for building APIs with Python based on standard Python type hints. It is designed for maximum speed and efficiency, offering performance comparable to Node.js and Go thanks to its native support for asynchronous programming. Key features include automatic interactive API documentation and robust data validation, allowing developers to write production-ready code quickly and with fewer bugs.', 'extras': {'signature': 'EoYTCoMTARFNMg+li1RFfgGmKlUjil86u1dji6Hx90OgOqW/xpcYSoifr/DjpRhX5vQ13RY72IgdE8NgVk8p8rpquWC1SeheSe72wx1ZA3ikZ89+/dlm0PX9v1v+wlU9jCXLhyag1L8/32tzpqsKrAg1G5c25iOe8XB/xH7k6eMiX3Eg+VhdULVU9M1470x/tFuIrCfChaTZHmfGlsWN0Va1UFCryU9bwNNOWuFpP8eeKdAEGgd1TQhsbhn0ktjTQHRyR/6JkH/iTbNNoiNcXNofZwu5ZK8huWBaDZJIrJ2XZOZfCedrsGBOs2dBfxsxy9ahN0BOY69UWAkE0pjDknSxgT6SUXNl+e0cm063J9YQ+d/vx8QCUKCgOmElVAR0/hIXOYH52yM9pXX1WisEeg6FtuKAQYuX2o6ZHnrpmNlk3f1MbZxpyUlnROPSKqKu1YV69E0ajuG+nnJ5bSVljNzJrqnhNKxRPS95P8/MwGABdFqF

Build a Token + Cost Analyzer

In [11]:
def analyze_usage(response):
  usage = response.usage_metadata

  input_tokens = usage["input_tokens"]
  output_tokens = usage["output_tokens"]
  total_tokens = usage["total_tokens"]

  print("Input Tokens:", input_tokens)
  print("Output Tokens: ", output_tokens)
  print("Total Tokens: ", total_tokens)

  return usage

In [12]:
usage = analyze_usage(response)

Input Tokens: 10
Output Tokens:  581
Total Tokens:  591


Estimate cost at scale

In [13]:
input_tokens_per_query = 2000
output_tokens_per_query = 500

input_price = 0.50
output_price = 1.50

def estimate_cost(num_queries):
  imput_cost = (
      num_queries
      * input_tokens_per_query
      / 1_000_000
      *input_price
  )

  output_cost = (
      num_queries
      * output_tokens_per_query
      / 1_000_000
      *output_price
  )
  total_cost = input_cost + output_cost

  print("Input cost: $", input_cost)
  print("Output cost: $", output_cost)
  print("Total cost: $", total_cost)

  return input_cost, output_cost, total_cost

In [16]:
for queries in [100,1000,10000]:
  input_cost, output_cost, total_cost = estimate_cost(queries)

  print(f"\nQueries: {queries}")
  print(f"Input cost:  ${input_cost:.4f}")
  print(f"Output cost: ${output_cost:.4f}")
  print(f"Total cost:  ${total_cost:.4f}")

Input cost: $ 0.001
Output cost: $ 0.07500000000000001
Total cost: $ 0.07600000000000001

Queries: 100
Input cost:  $0.0010
Output cost: $0.0750
Total cost:  $0.0760
Input cost: $ 0.001
Output cost: $ 0.75
Total cost: $ 0.751

Queries: 1000
Input cost:  $0.0010
Output cost: $0.7500
Total cost:  $0.7510
Input cost: $ 0.001
Output cost: $ 7.5
Total cost: $ 7.501

Queries: 10000
Input cost:  $0.0010
Output cost: $7.5000
Total cost:  $7.5010


See how context affects token usage

In [17]:
short_text = """
FastAPI is a Python web framework for building APIs.
"""

long_text = short_text * 50

In [18]:
short_response = llm.invoke(
    f"Summarize this text in one sentence:\n\n{short_text}"
)

long_response = llm.invoke(
    f"Summarize this text in one sentence:\n\n{long_text}"
)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [20]:
print("SHORT:")
print(short_response.usage_metadata)

print("\nLONG:")
print(long_response.usage_metadata)

SHORT:
{'input_tokens': 22, 'output_tokens': 272, 'total_tokens': 294, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 261}}

LONG:
{'input_tokens': 610, 'output_tokens': 224, 'total_tokens': 834, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 212}}
